# 图片降噪小工具

这是一个可以直接在 Jupyter Notebook 里运行的图片降噪小工具，自带一个简单的交互界面（用 `ipywidgets` 实现），效果类似一个小网页：

1. 填写图片文件路径（可以把文件直接拖进输入框）
2. 选择降噪方法和强度
3. 点击「开始降噪」查看效果
4. 点击下载链接保存结果

依次运行下面每一个代码格（Shift + Enter）即可，第一次运行需要联网安装依赖库。

In [ ]:
# 第一步：安装依赖库（第一次运行需要，之后可以跳过这一格）
!pip install opencv-python-headless pillow numpy matplotlib ipywidgets --quiet

In [ ]:
# 第二步：导入需要用到的库
%matplotlib inline
import io
import base64

import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

## 降噪方法说明

- **高斯模糊（Gaussian）**：速度最快，对轻微噪点效果不错，但会让图片整体变模糊。
- **中值滤波（Median）**：对"椒盐噪点"（图上一些孤立的黑白点）效果特别好。
- **双边滤波（Bilateral）**：在降噪的同时尽量保留边缘细节，速度中等。
- **非局部均值（Non-Local Means）**：效果通常最好，细节保留最完整，但处理速度比较慢。

"强度"数值越大，降噪效果越强，但也越容易丢失细节，建议先从中间值开始尝试。

In [ ]:
# 第三步：定义降噪函数
def denoise_image(img_bgr, method, strength):
    strength = int(strength)
    if method == 'gaussian':
        k = strength * 2 + 1  # 保证是奇数
        return cv2.GaussianBlur(img_bgr, (k, k), 0)
    elif method == 'median':
        k = strength * 2 + 1  # 保证是奇数
        return cv2.medianBlur(img_bgr, k)
    elif method == 'bilateral':
        d = max(3, strength)
        return cv2.bilateralFilter(img_bgr, d, strength * 10, strength * 10)
    elif method == 'nlm':
        h = max(1, strength * 2)
        return cv2.fastNlMeansDenoisingColored(img_bgr, None, h, h, 7, 21)
    else:
        raise ValueError(f'未知方法: {method}')

In [ ]:
# 第四步：构建交互界面并显示（运行这一格后，界面会出现在下方）

import os
from datetime import datetime

path_widget = widgets.Text(
    value='',
    placeholder='把图片文件拖进这个框，或粘贴完整路径，例如 /Users/你的用户名/Desktop/photo.png',
    description='图片路径:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px')
)

method_widget = widgets.Dropdown(
    options=[
        ('高斯模糊 - 速度快，适合轻微噪点', 'gaussian'),
        ('中值滤波 - 适合椒盐噪点', 'median'),
        ('双边滤波 - 兼顾降噪与边缘保留', 'bilateral'),
        ('非局部均值 - 效果最好，速度较慢', 'nlm'),
    ],
    value='nlm',
    description='方法:'
)

strength_widget = widgets.IntSlider(value=5, min=1, max=15, step=1, description='强度:')

process_button = widgets.Button(description='开始降噪', button_style='success')

output_folder_widget = widgets.Text(
    value='',
    placeholder='填写保存的文件夹路径，例如 /Users/你的用户名/Desktop',
    description='保存到:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='600px')
)

save_button = widgets.Button(description='保存到这个文件夹', button_style='info')

output_area = widgets.Output()

# 用来在"开始降噪"和"保存"两个按钮之间传递处理结果
_last_result = {'image': None}


def on_process_click(_):
    with output_area:
        clear_output(wait=True)

        img_path = path_widget.value.strip().strip('"').strip("'")
        if not img_path:
            print('请先在上面的框里填写图片路径')
            return
        if not os.path.exists(img_path):
            print(f'找不到这个文件: {img_path!r}')
            print('提示：Mac 上可以在 Finder 里选中文件，按住 option 键右键点击，选择"拷贝'
                  ' "文件名" 为路径名"；Windows 上可以在文件资源管理器里按住 Shift 右键文件，选择"复制为路径"。')
            return

        img = Image.open(img_path).convert('RGB')
        img_np = np.array(img)
        img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)

        result_bgr = denoise_image(img_bgr, method_widget.value, strength_widget.value)
        result_rgb = cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB)

        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        axes[0].imshow(img_np)
        axes[0].set_title('Original')
        axes[0].axis('off')
        axes[1].imshow(result_rgb)
        axes[1].set_title('Denoised')
        axes[1].axis('off')
        plt.tight_layout()
        plt.show()

        result_img = Image.fromarray(result_rgb)
        _last_result['image'] = result_img

        buf = io.BytesIO()
        result_img.save(buf, format='PNG')
        b64 = base64.b64encode(buf.getvalue()).decode()
        href = (
            f'<a download="denoised.png" '
            f'href="data:image/png;base64,{b64}" '
            f'style="display:inline-block;margin-top:10px;padding:8px 16px;'
            f'background:#2e7d32;color:white;border-radius:4px;'
            f'text-decoration:none;">点击下载降噪后的图片（浏览器下载）</a>'
        )
        display(HTML(href))
        print('处理完成，如果想直接存到指定文件夹，在下面填好路径后点"保存到这个文件夹"。')


def on_save_click(_):
    with output_area:
        if _last_result['image'] is None:
            print('还没有处理结果，请先点"开始降噪"。')
            return

        folder = output_folder_widget.value.strip().strip('"').strip("'")
        if not folder:
            print('请先在"保存到"框里填写文件夹路径')
            return
        if not os.path.isdir(folder):
            print(f'这个文件夹不存在: {folder!r}，请检查路径是否正确。')
            return

        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        save_path = os.path.join(folder, f'denoised_{timestamp}.png')
        _last_result['image'].save(save_path)
        print(f'已保存到: {save_path}')


process_button.on_click(on_process_click)
save_button.on_click(on_save_click)

display(widgets.VBox([
    path_widget,
    method_widget,
    strength_widget,
    process_button,
    output_folder_widget,
    save_button,
    output_area
]))

## 小提示

- 如果界面没有正常显示（比如按钮点不动、看不到上传框），大概率是 `ipywidgets` 没有正确启用。可以尝试重启 Jupyter 内核（Kernel → Restart），或者在终端运行：
  `jupyter nbextension enable --py widgetsnbextension`（经典 Notebook）——较新版本的 JupyterLab / Notebook 通常不需要这一步。
- 图片较大时，「非局部均值」方法会比较慢，可以先用「高斯模糊」或「中值滤波」快速预览效果。
- 如果想批量处理很多张图片（比如一个文件夹），可以告诉我，我再帮你写一个批处理版本的代码。